# 03 FLUX — TRIM VAE trening (Var 1: FLUX + TCR)

**Razsiritev diplome:** metabolni flux (scFEA, 168 modulov) ZAMENJA RNA kot modaliteta.
Model je isti TRIM VAE, le RNA veja -> FLUX veja (encoder/decoder). TCR del NEDOTAKNJEN.

Vprasanje: ali flux sam (brez RNA) nosi dovolj informacije za napoved klonalne ekspanzije?
Primerjava: Flux+TCR AUC vs RNA+TCR AUC (0.87). Kombinacijo vseh treh (Var 2) naredimo kasneje.

Vhod: `data_flux.pkl` (69930 x 168, iz notebooks_flux/01). Heldout = P24 (isti kot RNA+TCR).

## 0. Namestitev in mount

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Konfiguracija

Nastavi parametre tukaj pred zagonom.

In [ ]:
# === POTI ===
data_parent_folder   = '/content/drive/MyDrive/Diploma/data/processed'
tcr_ae_train_step    = 50100

# === TRENING ===
heldout_patient      = [24]    # indeks pacienta ki ga izpustimo (0-indexed)
                               # P24 ima 8152 Tumor-Post celic + vse 4 kvadrante
                               # (P0 nima tumorja -> neuporaben za evalvacijo)
training_steps       = 20000
batch_size           = 4096
n_learn_emb_sample   = 5120
lr                   = 0.001
print_every          = 100
save_every           = 10000
device_str           = 'cuda:0'

# === ARHITEKTURA ===
learn_patient_embeddings = 1
n_channels_base      = 2048
dimz                 = 1024
dim_state_embedding  = 128
# reduction: FLUX gre BREZ PCA (168 dim je ze nizko; moduli = smiselne enote)

# === LOSS ===
lambda_kl            = 15
lambda_recon_flux    = 1
lambda_recon_tcr     = 1
lambda_embedding_norm = 10
delta_contrastive    = 10

# === MONTE CARLO SEED ===
seed                 = 0      # spremeni za ponovitve (0,1,2). Fiksira init+shuffle za reproducibilnost.
flux_seed            = seed   # sparovano: flux_seed = TRIM seed

print('Konfiguracija nastavljena (Var 1: FLUX + TCR).')

## 2. Uvozi

In [ ]:
import numpy as np
import pandas as pd
import sklearn.decomposition
import sklearn.metrics
import matplotlib as mpl
import matplotlib.pyplot as plt
import umap
import time
import os
import json
import pickle
import torch
from torch import nn
import torch.nn.functional as F

# Izpeljane poti
tcr_folder   = os.path.join(data_parent_folder, f'tcr_ae/step_{tcr_ae_train_step}')
# FLUX: brez PCA; UMAP na fluxu (lasten cache); rezultati v LOCENO mapo (ne prepise RNA)
umap_file    = os.path.join(data_parent_folder, 'umap_trained_flux.pkl')
# NOVA HIERARHIJA: runs/P{pid}/seed{s}/flux_tcr/ (deljeni flux/rna/tcr centralno)
pid_str = '_'.join(map(str, heldout_patient))
output_folder = os.path.join(data_parent_folder, 'runs', f'P{pid_str}', f'seed{seed}', 'flux_tcr')
os.makedirs(output_folder, exist_ok=True)

device = torch.device(device_str if torch.cuda.is_available() else 'cpu')

# reproducibilnost: fiksiraj vse random vire (sicer trening nedeterministicen)
torch.manual_seed(seed); np.random.seed(seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
print(f'Seed: {seed} (reproducibilno)')

print(f'TCR folder:    {tcr_folder}')
print(f'Output folder: {output_folder}')
print(f'Device:        {device}')

fig = plt.figure()

## 3. Pomožne funkcije in razredi

In [ ]:
class Loader(object):
    def __init__(self, data, shuffle=False):
        self.start = 0
        self.epoch = 0
        self.data = data if type(data) == list else [data]
        if shuffle:
            self.r = list(range(data[0].shape[0]))
            np.random.shuffle(self.r)
            self.data = [x[self.r] for x in self.data]

    def next_batch(self, batch_size=100):
        num_rows = self.data[0].shape[0]
        if self.start + batch_size < num_rows:
            batch = [x[self.start:self.start + batch_size] for x in self.data]
            self.start += batch_size
        else:
            self.epoch += 1
            batch_part1 = [x[self.start:] for x in self.data]
            batch_part2 = [x[:batch_size - (x.shape[0] - self.start)] for x in self.data]
            batch = [np.concatenate([x1, x2], axis=0) for x1, x2 in zip(batch_part1, batch_part2)]
            self.start = batch_size - (num_rows - self.start)
        return batch if len(self.data) > 1 else batch[0]

    def iter_batches(self, batch_size=100):
        num_rows = self.data[0].shape[0]
        end = 0
        if batch_size > num_rows:
            yield self.data if len(self.data) > 1 else self.data[0]
        else:
            for i in range(num_rows // batch_size):
                start, end = i * batch_size, (i + 1) * batch_size
                batch = [x[start:end] for x in self.data]
                yield batch if len(self.data) > 1 else batch[0]
            if end < num_rows:
                batch = [x[end:] for x in self.data]
                yield batch if len(self.data) > 1 else batch[0]


def numpy2torch(x, type=torch.FloatTensor):
    return torch.from_numpy(x).type(type).to(device)


def reparameterize(mu, logvar):
    std = logvar.mul(0.5).exp_()
    eps = torch.autograd.Variable(std.data.new(std.size()).normal_())
    return eps.mul(std).add_(mu)


def check_for_nan(model):
    for name, param in model.named_parameters():
        if param.grad is not None and torch.isnan(param.grad).any():
            print('NaN v:', name)
            raise Exception('NaN v gradientih!')


def make_legend(ax, labels, s=20, cmap=mpl.cm.jet, **kwargs):
    numlabs = len(labels)
    for i, label in enumerate(labels):
        c = [cmap(1 * i / (numlabs - 1))] if numlabs > 1 else [cmap(1.)]
        ax.scatter(0, 0, s=s, c=c, label=label)
    ax.scatter(0, 0, s=2*s, c='w')
    ax.legend(**kwargs)


def scatter_helper(embedding1, embedding2, ax, **kwargs):
    e = np.concatenate([embedding1, embedding2], axis=0)
    l = np.concatenate([np.zeros(embedding1.shape[0]), np.ones(embedding2.shape[0])], axis=0)
    r = np.random.choice(range(e.shape[0]), e.shape[0], replace=False)
    if e.shape[0] > 0:
        ax.scatter(e[r, 0], e[r, 1], c=l[r], s=1, **kwargs)


print('Pomožne funkcije definirane.')

## 4. Arhitektura modela (MLP + Generator) — FLUX veja namesto RNA

In [ ]:
class MLP(nn.Module):
    """4-slojni MLP z BatchNorm in LeakyReLU. decoder=True obrne velikosti slojev."""
    def __init__(self, **kwargs):
        super().__init__()
        nbase = kwargs['nbase']
        dim_in = kwargs['dim_in']
        dim_out = kwargs['dim_out']
        layers = [1, 2, 4]
        if 'decoder' in kwargs:
            layers = layers[::-1]
        self.layer1 = nn.Linear(dim_in, nbase // layers[0])
        self.layer2 = nn.Linear(nbase // layers[0], nbase // layers[1])
        self.layer3 = nn.Linear(nbase // layers[1], nbase // layers[2])
        self.out    = nn.Linear(nbase // layers[2], dim_out)
        self.bn1 = nn.BatchNorm1d(nbase // layers[0])
        self.bn2 = nn.BatchNorm1d(nbase // layers[1])
        self.bn3 = nn.BatchNorm1d(nbase // layers[2])
        self.act = kwargs.get('act', torch.nn.LeakyReLU())

    def forward(self, x):
        h1 = self.act(self.bn1(self.layer1(x)))
        h2 = self.act(self.bn2(self.layer2(h1)))
        h3 = self.act(self.bn3(self.layer3(h2)))
        return self.out(h3)


class Generator(nn.Module):
    """
    TRIM VAE:
    - FLUX encoder + TCR encoder → fused z (povprečje mu/logvar)
    - FLUX decoder + TCR decoder
    - Conditioning: tissue, prepost, patient (vsak 128-dim, learnable)
    """
    def __init__(self):
        super().__init__()
        nbase = n_channels_base
        n_cond = 3 * dim_state_embedding

        self.encoder_flux = MLP(dim_in=dimflux + n_cond, dim_out=dimz * 2, nbase=nbase)
        self.decoder_flux = MLP(dim_in=dimz + n_cond,   dim_out=dimflux,  nbase=nbase, decoder=True)
        self.encoder_tcr = MLP(dim_in=dimtcr + n_cond, dim_out=dimz * 2, nbase=nbase)
        self.decoder_tcr = MLP(dim_in=dimz + n_cond,   dim_out=dimtcr,   nbase=nbase, decoder=True)

        self.register_buffer('bloodtumor_embeddings_matrix', torch.zeros(2, dim_state_embedding))
        self.register_buffer('prepost_embeddings_matrix',    torch.zeros(2, dim_state_embedding))
        self.register_buffer('patient_embeddings_matrix',    torch.zeros(num_patients, dim_state_embedding))

        self.mlp_patient_embeddings_flux = MLP(dim_in=dimflux, dim_out=dim_state_embedding, nbase=nbase)
        self.mlp_patient_embeddings_tcr = MLP(dim_in=dimtcr, dim_out=dim_state_embedding, nbase=nbase)
        self.lrelu = torch.nn.LeakyReLU()

    def forward(self, x, embeddings):
        x_flux = torch.cat([x[0]] + embeddings, axis=-1)
        x_tcr = torch.cat([x[1]] + embeddings, axis=-1)
        z_flux = self.encoder_flux(x_flux)
        z_tcr = self.encoder_tcr(x_tcr)
        mu     = torch.mean(torch.stack([z_flux[:, :dimz], z_tcr[:, :dimz]]), 0)
        logvar = torch.mean(torch.stack([z_flux[:, dimz:], z_tcr[:, dimz:]]), 0)
        z = reparameterize(mu, logvar)
        shared = torch.cat([z] + embeddings, axis=-1)
        return self.decoder_flux(shared), self.decoder_tcr(shared), [mu, logvar, [z_flux, z_tcr]]

    def sample(self, z, embeddings):
        shared = torch.cat([z] + embeddings, axis=-1)
        return self.decoder_flux(shared), self.decoder_tcr(shared), [None, None, shared]

    def groupby_mean(self, value, labels, num_labels):
        for i_pid in range(num_labels):
            if (labels == i_pid).sum() == 0:
                value  = torch.cat([value, torch.zeros_like(value[0])[np.newaxis, :]], axis=0)
                labels = torch.cat([labels, torch.Tensor([i_pid]).to(device)], axis=0)
        uniques = labels.unique().tolist()
        key_val = {key: val for key, val in zip(uniques, range(len(uniques)))}
        val_key = {int(val): int(key) for key, val in zip(uniques, range(len(uniques)))}
        labels_idx = torch.LongTensor(list(map(key_val.get, labels.tolist()))).to(device)
        labels_idx = labels_idx.view(labels_idx.size(0), 1).expand(-1, value.size(1))
        unique_labels, labels_count = labels_idx.unique(dim=0, return_counts=True)
        result = torch.zeros_like(unique_labels, dtype=torch.float).scatter_add_(0, labels_idx, value)
        result = result / labels_count.float().unsqueeze(1)
        new_labels = torch.LongTensor(list(map(val_key.get, unique_labels[:, 0].type(torch.int32).tolist())))
        return result, new_labels


print('Arhitektura definirana.')

## 5. Naloži podatke (FLUX namesto RNA)

In [ ]:
t = time.time()

with open(os.path.join(data_parent_folder, 'data_flux.pkl'), 'rb') as f:
    data_flux = pickle.load(f)
print(f'data_flux: {data_flux.shape}')

with open(os.path.join(tcr_folder, 'data_tcr.pkl'), 'rb') as f:
    data_tcr = pickle.load(f)
print(f'data_tcr: {data_tcr.shape}')

with open(os.path.join(data_parent_folder, 'data_labels.pkl'), 'rb') as f:
    data_labels = pickle.load(f)

with open(os.path.join(data_parent_folder, 'data_labels_str.pkl'), 'rb') as f:
    data_labels_str = pickle.load(f)

with open(os.path.join(data_parent_folder, 'df_all_tcrs.pkl'), 'rb') as f:
    df_all_tcrs = pickle.load(f)

# Stolpci
col_bloodtumor = data_labels.columns.get_loc('Tissue')
col_prepost    = data_labels.columns.get_loc('Treatment Stage')
col_celltype   = data_labels.columns.get_loc('SubCellType')
col_patient    = data_labels.columns.get_loc('Patient')
col_tcr        = data_labels.columns.get_loc('CDR3(Beta1)')
ID_NULL_TCR    = -1

print(f'data_labels: {data_labels.shape}, stolpci: {data_labels.columns.tolist()}')
print(f'Podatki naloženi v {time.time()-t:.1f} s')

## 6. UMAP na fluxu (BREZ PCA — 168 dim je ze nizko)

In [ ]:
# FLUX gre BREZ PCA: 168 dimenzij je ze nizko (RNA je bil 36k genov -> PCA 100).
# Moduli so ze smiselne enote; PCA bi zabrisal interpretabilnost.
data_flux = np.asarray(data_flux, dtype=np.float32)
assert data_flux.shape[0] == data_labels.shape[0], 'data_flux ni poravnan z data_labels!'

# === Z-STANDARDIZACIJA PO MODULIH (axis=0) ===
# Zakaj: scFEA flux je v MAJHNI skali (mean~0.006) -> MSE recon loss ~0.0001 ->
# v skupnem loss-u utopljen ob TCR (~0.15-0.3) -> model flux IGNORIRA (flux=0.000).
# RNA tega ni rabil: normalizacija ([0.84,1]) + PCA na visoko-variabilnih genih da
# komponente reda ~enot. Flux nima ne enega ne drugega -> eksplicitno standardiziramo.
# Po modulih (vsak stolpec svoj mean/std) -> vsak modul std=1, primerljivo s TCR.
# REVERZIBILNO: shranimo mean/std -> kadarkoli nazaj v prave flukse (flux_z*std+mean).
flux_mean = data_flux.mean(axis=0)
flux_std  = data_flux.std(axis=0) + 1e-8           # +eps: mrtev modul (std=0) ne deli z nic
print(f'FLUX skala PRED: mean(x^2)={(data_flux**2).mean():.6f}  (MSE ~0.000 -> flux se ne uci)')
data_flux = (data_flux - flux_mean) / flux_std
print(f'FLUX skala PO:   mean(x^2)={(data_flux**2).mean():.4f}  (~1.0 -> primerljivo s TCR)')

# shrani mean/std za reverzibilnost (pretvorba napovedi nazaj v prave flukse)
with open(os.path.join(data_parent_folder, 'flux_standardize.pkl'), 'wb') as f:
    pickle.dump({'mean': flux_mean, 'std': flux_std}, f)
print(f'FLUX (standardiziran): {data_flux.shape} | mean/std shranjena za reverzibilnost')

# UMAP za vizualizacijo (na STANDARDIZIRANEM fluxu = kar gre v model, lasten cache)
if not os.path.exists(umap_file):
    viz_reducer = umap.UMAP(random_state=0).fit(data_flux)
    pickle.dump(viz_reducer, open(umap_file, 'wb+'))
    print('UMAP (flux) naucen in shranjen.')
else:
    viz_reducer = pickle.load(open(umap_file, 'rb'))
    print('UMAP (flux) nalozen iz cache-a.')

e_eval_reals = viz_reducer.transform(data_flux)
print(f'UMAP embeddingi: {e_eval_reals.shape}')

## 7. Pripravi podatke za trening (FLUX + TCR)

In [ ]:
x_flux  = data_flux
x_tcr   = data_tcr
x_label = data_labels.values

# Vse celice morajo imeti TCR (pri nas so že filtrirane)
mask_preprocess = x_label[:, col_tcr] != ID_NULL_TCR
assert mask_preprocess.sum() == x_flux.shape[0], 'Nekatere celice nimajo TCR!'

dimflux      = x_flux.shape[-1]
dimtcr       = x_tcr.shape[-1]
num_patients = len(np.unique(x_label[:, col_patient]))

print(f'FLUX dim: {dimflux}, TCR dim: {dimtcr}, Pacientov: {num_patients}')
assert x_label[:, col_patient].max() == num_patients - 1, 'Patient ID-ji niso 0-indexed!'
assert np.isin(x_label[:, col_patient], heldout_patient).sum() > 0, 'Heldout pacient ni v podatkih!'

# --- PREVERBA HELDOUT PACIENTA: razclenitev po 4 kvadrantih ---
# TRIM napoveduje Tumor-Post; heldout MORA imeti dovolj tumor-post celic za smiselno evalvacijo.
print('\n=== Heldout pacient(i) po kvadrantih ===')
for pid in heldout_patient:
    m = x_label[:, col_patient] == pid
    q = {}
    for t, tn in [(0,'Blood'),(1,'Tumor')]:
        for s, sn in [(0,'Pre'),(1,'Post')]:
            q[f'{tn}-{sn}'] = int((m & (x_label[:,col_bloodtumor]==t) & (x_label[:,col_prepost]==s)).sum())
    print(f'  P{pid}: {q}')
    if q['Tumor-Post'] < 50:
        print(f'  ⚠️ OPOZORILO: P{pid} ima le {q["Tumor-Post"]} Tumor-Post celic — premalo za evalvacijo! Zamenjaj heldout.')
# Predlog: kateri pacienti imajo najvec Tumor-Post celic
tp_counts = {int(pid): int(((x_label[:,col_patient]==pid)&(x_label[:,col_bloodtumor]==1)&(x_label[:,col_prepost]==1)).sum())
             for pid in np.unique(x_label[:,col_patient])}
top = sorted(tp_counts.items(), key=lambda kv: -kv[1])[:6]
print(f'  Pacienti z najvec Tumor-Post celic: {top}')

# Trening maska: vse celice razen heldout, RAZEN blood-pre od heldout (te so vhod za counterfactual)
mask_train = np.logical_or(
    ~np.isin(x_label[:, col_patient], heldout_patient),
    np.logical_and(x_label[:, col_bloodtumor] == 0, x_label[:, col_prepost] == 0)
)
print(f'\nTrening celic: {mask_train.sum()} / {len(mask_train)}')
print(f'Heldout celic: {(~mask_train).sum()}')

## 8. Inicializacija modela

In [ ]:
G = Generator()
G = G.to(device)

param_list = list(G.parameters())
opt_G = torch.optim.Adam(param_list, lr=lr)

load_train = Loader([x_flux[mask_train], x_tcr[mask_train], x_label[mask_train]], shuffle=True)
load_eval  = Loader([x_flux, x_tcr, x_label], shuffle=False)

# Batch za učenje kondicioniranih embeddingov (enkrat vzamemo in fiksiramo)
learn_emb_batch_x_flux, learn_emb_batch_tcr, learn_emb_batch_labels = load_train.next_batch(n_learn_emb_sample)
learn_emb_batch_x_flux   = numpy2torch(learn_emb_batch_x_flux)
learn_emb_batch_tcr      = numpy2torch(learn_emb_batch_tcr)
learn_emb_batch_labels   = numpy2torch(learn_emb_batch_labels)
learn_emb_mask_bb = torch.logical_and(
    learn_emb_batch_labels[:, col_bloodtumor] == 0,
    learn_emb_batch_labels[:, col_prepost] == 0
)

n_params = sum(p.numel() for p in G.parameters())
print(f'Model parametrov: {n_params:,}')
print(f'Batch size: {batch_size}, Training steps: {training_steps}')

## 9. Trening

Loss ima 4 komponente:
- **KL** (λ=15): regularizacija latentnega prostora
- **MSE FLUX** (λ=1): rekonstrukcija metabolnega fluxa (standardiziran -> loss primerljiv s TCR)
- **Contrastive TCR** (λ=1): iste klone potisni skupaj, različne narazen
- **Embedding norm** (λ=10): prepreči eksplozijo kondicioniranih embeddingov

Spremljaj razčlenjeni loss (vsakih 1000 korakov): **flux=** mora biti zdaj v istem
redu velikosti kot **tcr_same/tcr_diff** (~0.1-0.3). Če je še vedno ~0.000, dvigni λ.

Pričakovano ~15 min na A100.

In [ ]:
i_iter = 0
losses = []
all_losses = []   # akumulator VSEH loss (losses se resetira pri printu)
t = time.time()

while i_iter <= training_steps:
    i_iter += 1
    G.train()
    opt_G.zero_grad()

    batch_x_flux, batch_x_tcr, batch_labels = load_train.next_batch(batch_size)
    batch_x_flux  = numpy2torch(batch_x_flux)
    batch_x_tcr   = numpy2torch(batch_x_tcr)
    batch_labels  = numpy2torch(batch_labels, type=torch.IntTensor)

    # Posodobi kondicionirane embeddinге iz fiksnega batch-a
    batch_emb_flux = G.mlp_patient_embeddings_flux(learn_emb_batch_x_flux)
    batch_emb_tcr = G.mlp_patient_embeddings_tcr(learn_emb_batch_tcr)
    batch_emb     = torch.stack([batch_emb_flux, batch_emb_tcr]).mean(axis=0)

    G.bloodtumor_embeddings_matrix = G.groupby_mean(batch_emb, learn_emb_batch_labels[:, col_bloodtumor], 2)[0]
    G.prepost_embeddings_matrix    = G.groupby_mean(batch_emb, learn_emb_batch_labels[:, col_prepost], 2)[0]
    G.patient_embeddings_matrix    = G.groupby_mean(batch_emb[learn_emb_mask_bb], learn_emb_batch_labels[learn_emb_mask_bb, col_patient], num_patients)[0]

    bt_emb  = torch.index_select(G.bloodtumor_embeddings_matrix, 0, batch_labels[:, col_bloodtumor].type(torch.int32))
    pp_emb  = torch.index_select(G.prepost_embeddings_matrix,    0, batch_labels[:, col_prepost].type(torch.int32))
    pat_emb = torch.index_select(G.patient_embeddings_matrix,    0, batch_labels[:, col_patient].type(torch.int32))

    recon_flux, recon_tcr, [mu, logvar, _] = G(x=[batch_x_flux, batch_x_tcr], embeddings=[bt_emb, pp_emb, pat_emb])

    # Loss
    kl = -(1 + logvar - logvar.exp() - mu.pow(2)).mean()
    loss_flux = ((batch_x_flux - recon_flux)**2).mean()

    real_same  = batch_labels[:, col_tcr][np.newaxis, :] == batch_labels[:, col_tcr][:, np.newaxis]
    real_diff  = ~real_same
    real_same  = torch.logical_and(real_same, torch.eye(batch_size).to(device) == 0)
    tcr_dists  = torch.cdist(recon_tcr, recon_tcr, p=1)
    loss_tcr_same = tcr_dists[real_same].mean()
    loss_tcr_diff = F.relu(delta_contrastive - tcr_dists[real_diff]).mean()

    loss_emb_norm = torch.cat([G.bloodtumor_embeddings_matrix**2,
                               G.prepost_embeddings_matrix**2,
                               G.patient_embeddings_matrix**2]).mean()

    batch_loss_list = [
        lambda_kl * kl,
        lambda_recon_flux * loss_flux,
        lambda_recon_tcr * loss_tcr_same,
        lambda_recon_tcr * loss_tcr_diff,
        lambda_embedding_norm * loss_emb_norm,
    ]
    total_loss = torch.mean(torch.stack(batch_loss_list))
    losses.append(total_loss.item()); all_losses.append(total_loss.item())

    total_loss.backward()
    check_for_nan(G)
    opt_G.step()
    opt_G.zero_grad()

    if i_iter % print_every == 0:
        print('{:>5}: avg loss: {:.3f} ({:.1f} s)'.format(i_iter, np.mean(losses), time.time() - t))
        if i_iter % (10 * print_every) == 0:
            names = ['kl', 'flux', 'tcr_same', 'tcr_diff', 'emb_norm']
            vals  = ['{:.3f}'.format(l.detach().cpu().numpy()) for l in batch_loss_list]
            print('       ' + '  '.join(f'{n}={v}' for n, v in zip(names, vals)))
        t = time.time()
        losses = []

    if i_iter % save_every == 0:
        torch.save(G.state_dict(), os.path.join(output_folder, 'model.pth'))
        torch.save(G.state_dict(), os.path.join(output_folder, f'model_step{i_iter}.pth'))  # checkpoint
        print(f'Model shranjen pri koraku {i_iter}.')

print('Trening končan!')

## 10. Napovedi (post-trening)

In [ ]:
def get_model_predictions(G, load_eval):
    G.eval()
    preds_flux, preds_tcr, recon_flux_z, recon_tcr_z = [], [], [], []

    # Fiksiramo embeddinге
    emb_flux = G.mlp_patient_embeddings_flux(learn_emb_batch_x_flux)
    emb_tcr = G.mlp_patient_embeddings_tcr(learn_emb_batch_tcr)
    emb = torch.stack([emb_flux, emb_tcr]).mean(axis=0)
    G.bloodtumor_embeddings_matrix = G.groupby_mean(emb, learn_emb_batch_labels[:, col_bloodtumor], 2)[0]
    G.prepost_embeddings_matrix    = G.groupby_mean(emb, learn_emb_batch_labels[:, col_prepost], 2)[0]
    G.patient_embeddings_matrix    = G.groupby_mean(emb[learn_emb_mask_bb], learn_emb_batch_labels[learn_emb_mask_bb, col_patient], num_patients)[0]

    for batch_x_flux, batch_x_tcr, batch_labels in load_eval.iter_batches(batch_size=batch_size):
        batch_random = numpy2torch(np.random.normal(0, 1, [batch_x_flux.shape[0], dimz]))
        batch_x_flux = numpy2torch(batch_x_flux)
        batch_x_tcr  = numpy2torch(batch_x_tcr)
        batch_labels = numpy2torch(batch_labels, type=torch.IntTensor)

        bt  = torch.index_select(G.bloodtumor_embeddings_matrix, 0, batch_labels[:, col_bloodtumor].type(torch.int32))
        pp  = torch.index_select(G.prepost_embeddings_matrix,    0, batch_labels[:, col_prepost].type(torch.int32))
        pat = torch.index_select(G.patient_embeddings_matrix,    0, batch_labels[:, col_patient].type(torch.int32))

        _, _, [_, _, [z_flux, z_tcr]] = G(x=[batch_x_flux, batch_x_tcr], embeddings=[bt, pp, pat])
        out_flux, out_tcr, _ = G.sample(z=batch_random, embeddings=[bt, pp, pat])

        preds_flux.append(out_flux.detach().cpu().numpy())
        preds_tcr.append(out_tcr.detach().cpu().numpy())
        recon_flux_z.append(z_flux.detach().cpu().numpy())
        recon_tcr_z.append(z_tcr.detach().cpu().numpy())

    return (np.concatenate(preds_flux), np.concatenate(preds_tcr),
            np.concatenate(recon_flux_z), np.concatenate(recon_tcr_z))


print('Računam napovedi...')
t = time.time()
preds_flux, preds_tcr, recon_flux_z, recon_tcr_z = get_model_predictions(G, load_eval)
print(f'Napovedi izračunane v {time.time()-t:.1f} s')
print(f'preds_flux: {preds_flux.shape}')

## 11. Pseudo-kloni (TCR clustering)

In [ ]:
def get_pseudoclones(preds_tcr, x_label, train_mask, dists, thresh, tol=.025, max_tries=50):
    num_unique = len(np.unique(x_label[train_mask, col_tcr]))
    good, tries = False, 0
    while not good:
        pseudo_tcrs = -10 * np.ones(x_label.shape[0])
        curr_tcr_id = 0
        while (pseudo_tcrs == -10).sum() > 0:
            i = np.random.choice(np.argwhere(pseudo_tcrs == -10).flatten())
            mask = np.logical_and(dists[i] < thresh, pseudo_tcrs == -10)
            mask = np.logical_and(mask, x_label[:, col_bloodtumor] == x_label[i, col_bloodtumor])
            mask = np.logical_and(mask, x_label[:, col_patient]    == x_label[i, col_patient])
            pseudo_tcrs[mask] = curr_tcr_id
            curr_tcr_id += 1
        num_unique_preds = len(np.unique(pseudo_tcrs[train_mask]))
        if (1-tol)*num_unique <= num_unique_preds <= (1+tol)*num_unique:
            good = True
        else:
            thresh *= .95 if num_unique_preds < num_unique else 1.05
        tries += 1
        if tries >= max_tries:
            break
    print(f'Pseudo klonov: {num_unique_preds} (real: {num_unique}), thresh={thresh:.2f}, tries={tries}')
    return pseudo_tcrs, thresh


# Pseudo-kloni SAMO na heldout celicah.
# Polna tcr_dists (146k x 146k) = ~172 GB -> crash. Original to racuna na 2 TB stroju.
# get_pseudoclones maska tako ali tako omeji na isti patient+bloodtumor (dists[i] se primerja
# le znotraj heldout pacienta), zato je rezultat za heldout celice IDENTICEN, a brez 172 GB.
# preds.npz tako ali tako shrani le pseudo_tcrs[mask_leave_out].
print('Računam TCR razdalje (samo heldout celice)...')
mask_leave_out = np.isin(x_label[:, col_patient], heldout_patient)
n_ho = mask_leave_out.sum()
print(f'Heldout celic: {n_ho} -> tcr_dists matrika ~{n_ho**2 * 8 / 1e9:.1f} GB')

preds_tcr_ho = preds_tcr[mask_leave_out]
x_label_ho   = x_label[mask_leave_out]
# train_mask znotraj heldout: za kalibracijo thresh (heldout pacient ima blood-pre v train mnozici)
train_mask_ho = mask_train[mask_leave_out]

tcr_dists_ho = sklearn.metrics.pairwise_distances(preds_tcr_ho, preds_tcr_ho, metric='l1')
pseudo_tcrs_ho, thresh_fitted = get_pseudoclones(preds_tcr_ho, x_label_ho, train_mask_ho, tcr_dists_ho, thresh=delta_contrastive)

# Razsiri nazaj na polno dolzino (146k), da cell shranjevanja [mask_leave_out] deluje.
# Ne-heldout celice dobijo -10 (placeholder); shrani se itak le heldout del.
pseudo_tcrs = -10 * np.ones(x_label.shape[0])
pseudo_tcrs[mask_leave_out] = pseudo_tcrs_ho
print('Pseudo-kloni izračunani.')

## 12. Shrani rezultate

In [ ]:
# Model
torch.save(G.state_dict(), os.path.join(output_folder, 'model.pth'))
print('model.pth shranjen.')

# Argumenti
config = dict(data_parent_folder=data_parent_folder, heldout_patient=heldout_patient,
              seed=seed, variant='flux_tcr',
              flux_seed=flux_seed,
              training_steps=training_steps, batch_size=batch_size, dimz=dimz,
              dim_state_embedding=dim_state_embedding, lambda_kl=lambda_kl,
              lambda_recon_flux=lambda_recon_flux, lambda_recon_tcr=lambda_recon_tcr,
              lambda_embedding_norm=lambda_embedding_norm, delta_contrastive=delta_contrastive)
with open(os.path.join(output_folder, 'args.txt'), 'w+') as f:
    json.dump(config, f, indent=2)
print('args.txt shranjen.')

# Napovedi
mask_leave_out = np.isin(x_label[:, col_patient], heldout_patient)
with open(os.path.join(output_folder, 'preds.npz'), 'wb+') as f:
    np.savez(f,
        preds_flux=preds_flux[mask_leave_out],
        preds_tcr=preds_tcr[mask_leave_out],
        recon_flux_z=recon_flux_z[mask_leave_out],
        recon_tcr_z=recon_tcr_z[mask_leave_out],
        pseudo_tcrs=pseudo_tcrs[mask_leave_out],
        thresh_fitted=thresh_fitted,
        learn_emb_batch_x_flux=learn_emb_batch_x_flux.cpu().numpy(),
        learn_emb_batch_tcr=learn_emb_batch_tcr.cpu().numpy(),
        learn_emb_batch_labels=learn_emb_batch_labels.cpu().numpy())
print(f'preds.npz shranjen: preds_flux={preds_flux[mask_leave_out].shape}')
# --- losses (za graf konvergence) + heldout_meta (za reproducibilnost) ---
with open(os.path.join(output_folder, 'losses.pkl'), 'wb') as f:
    pickle.dump(all_losses if 'all_losses' in dir() else [], f)  # VSE loss, ne resetiranih
meta = {'pid': [int(p) for p in heldout_patient], 'seed': int(seed), 'variant': 'flux_tcr',
        'n_train_cells': int(mask_train.sum()), 'n_heldout_cells': int((~mask_train).sum())}
meta['flux_seed'] = int(flux_seed)
with open(os.path.join(output_folder, 'heldout_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('losses.pkl + heldout_meta.json shranjena.')

print(f'Vse shranjeno v: {output_folder}')

## 13. UMAP vizualizacija

In [ ]:
print('Računam UMAP napovedi...')
e_eval_preds = viz_reducer.transform(preds_flux)

for i_pid in range(num_patients):
    mask1 = np.logical_and(x_label[:, col_bloodtumor] == 0, x_label[:, col_prepost] == 0) & (x_label[:, col_patient] == i_pid)
    mask2 = np.logical_and(x_label[:, col_bloodtumor] == 0, x_label[:, col_prepost] == 1) & (x_label[:, col_patient] == i_pid)
    mask3 = np.logical_and(x_label[:, col_bloodtumor] == 1, x_label[:, col_prepost] == 0) & (x_label[:, col_patient] == i_pid)
    mask4 = np.logical_and(x_label[:, col_bloodtumor] == 1, x_label[:, col_prepost] == 1) & (x_label[:, col_patient] == i_pid)

    fig.clf()
    fig.suptitle(f'Pacient {i_pid}')
    axes = fig.subplots(2, 2, sharex=True, sharey=True)
    [make_legend(ax, ['Real', 'Predicted'], cmap=mpl.cm.viridis, fontsize=6) for ax in axes.flatten()]
    scatter_helper(e_eval_reals[mask1], e_eval_preds[mask1], ax=axes[0, 0])
    scatter_helper(e_eval_reals[mask2], e_eval_preds[mask2], ax=axes[0, 1])
    scatter_helper(e_eval_reals[mask3], e_eval_preds[mask3], ax=axes[1, 0])
    scatter_helper(e_eval_reals[mask4], e_eval_preds[mask4], ax=axes[1, 1])
    axes[0, 0].set_title('Blood Pre'); axes[0, 1].set_title('Blood Post')
    axes[1, 0].set_title('Tumor Pre'); axes[1, 1].set_title('Tumor Post')
    [[ax.set_xticks([]), ax.set_yticks([])] for ax in axes.flatten()]
    fig.tight_layout()
    fig.savefig(os.path.join(output_folder, f'umap_flux_patient{i_pid}.png'), dpi=150)
    if i_pid in heldout_patient:
        plt.show()

print('Vizualizacije shranjene.')
print('Vse končano!')